# Week 5 — Expanding the Urdu OCR Training Dataset

This notebook adds three new ways to grow `labels.csv` beyond Week 3's synthetic renders,
on top of the Week 4 audit's path-resolution fix (so every row this notebook writes uses
the *correct* path convention from the start — relative to `DATA_DIR`, not its parent).

**What this notebook does NOT do, and why:** the original ask for this pass included
scraping BBC Urdu, Jang, Akhbar-e-Jahan, and Rekhta. Those are commercial/copyrighted
platforms, and building a pipeline to systematically pull their text and scanned images
for a training corpus goes beyond personal use into the kind of bulk reproduction their
terms prohibit. Rekhta's own terms are explicit: *"You shall not download any Content
unless you see a 'download' link... You shall not copy, reproduce, make available online
or electronically transmit... any Content."* Their FAQ adds that contributors only
authorized Rekhta to digitize works for on-site reading, not redistribution — so this
isn't just a terms-of-service technicality, it's a copyright issue one level up too. BBC
and Jang carry the same basic concern as commercial news publishers.

Worth knowing separately: Week 3's existing `headlines.csv` corpus (137,105 rows, only
~110 of which have been rendered so far) contains malformed rows with embedded
`express.pk` URLs and GMT timestamps — it looks like it was itself scraped from a news
site at some point before landing in this repo. This notebook uses a small sample of it
below purely to test the renderer against real text (same as testing against any other
file already in the repo), not as a green light to render all 137K rows. Headline/title
text sits in a much smaller copyright gray area than full articles or scanned book pages —
short factual titles generally get little to no protection — but it's worth confirming
this file's origin with your program mentors before leaning on it at scale. For real
scaling, prefer a clean-provenance source: Wikipedia Urdu (CC BY-SA) for text, or one of
the purpose-built OCR datasets in the summary cell at the end (UPTI, MMU-OCR-21,
FIPU-OCR-CHAR, UrduDoc) — these also give better ground truth than any scraped web text,
since they're purpose-annotated rather than incidentally collected.

Three pipelines, all writing to the same `labels.csv` schema (`image`, `text`,
`category`), all producing paths relative to `DATA_DIR`:
1. **Expanded synthetic rendering** — multi-font, with realistic augmentation (rotation,
   blur, noise, paper-tone backgrounds) so the model doesn't only ever see perfectly clean
   renders.
2. **PDF extraction** — for PDFs *you have rights to use* that have a real text layer:
   crops the actual rendered page to each line's exact position, paired with the exact
   text. Real fonts/kerning/layout, not synthetic.
3. **DOCX extraction** — pulls paragraph text out of your own `.docx` files and feeds it
   through the same renderer as (1), since DOCX has no fixed visual layout to crop from.

Run Step 4 last regardless of which of 1–3 you use — it merges whatever new rows exist
into `labels.csv` and backs up the original first.

## Step 0: Setup

Same repo/paths as Week 4.

In [ ]:
import os

REPO_URL = "https://github.com/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git"
REPO_DIR = "/content/urdu-ocr-repo"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

DATA_DIR = os.path.join(REPO_DIR, "SI26-Week1", "data")
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
print("DATA_DIR:", DATA_DIR)
print("labels.csv exists:", os.path.isfile(LABELS_PATH))

In [ ]:
!pip install pdfplumber pypdfium2 python-docx --quiet
import pandas as pd
import numpy as np
print("Ready.")

## Step 1: Expanded Synthetic Rendering

Renders text to line images using Pillow's raqm/HarfBuzz-backed shaping
(`direction="rtl", language="ur"`) — the same approach Week 1/Week 3 already use, so
output is consistent with the rest of the project. Two things beyond what Week 3 did:

- **Multiple fonts** (`FONTS` below — both already in the repo; add more paths here if
  you have other properly-licensed Urdu fonts, e.g. more Noto/SIL-OFL-licensed families).
- **Augmentation** — mild rotation, blur, gaussian noise, and a varied paper-tone
  background instead of pure white, so the model sees some of the variation a scan or
  photo would introduce, instead of only ever training on perfectly clean synthetic text.

`make_dataset()` is corpus-agnostic: pass it any DataFrame with a text column. It's used
below with a *small* sample of the existing headlines corpus to prove it works end-to-end
— see the note in the intro cell about why this stays small rather than "render
everything." If you need to actually scale this, swap in a clean-provenance corpus (Wikipedia
Urdu, or the OCR-specific datasets in the summary cell) as the `corpus_df` argument.

In [ ]:
import random
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps

FONTS = {
    "nastaliq": os.path.join(REPO_DIR, "SI26-Week1", "NotoNastaliqUrdu-Regular.ttf"),
    "naskh": os.path.join(REPO_DIR, "SI26-Week3", "fonts", "NotoNaskhArabic-Variable.ttf"),
}


def render_line(text, font_path, font_size=44, padding=18):
    """Render one line of Urdu text to a clean image (raqm-backed RTL shaping)."""
    font = ImageFont.truetype(font_path, font_size)
    tmp = Image.new("L", (10, 10))
    d = ImageDraw.Draw(tmp)
    bbox = d.textbbox((0, 0), text, font=font, direction="rtl", language="ur")
    w = int(bbox[2] - bbox[0]) + 2 * padding
    h = int(bbox[3] - bbox[1]) + 2 * padding
    img = Image.new("RGB", (w, h), "white")
    draw = ImageDraw.Draw(img)
    draw.text((padding - bbox[0], padding - bbox[1]), text, font=font, fill="black",
               direction="rtl", language="ur")
    return img


def augment(img, rng):
    """Mild randomized degradations: paper-tone background, slight rotation, blur, noise."""
    tint = rng.randint(235, 255)
    bg = Image.new("RGB", img.size, (tint, tint - rng.randint(0, 8), tint - rng.randint(0, 15)))
    mask = ImageOps.invert(img.convert("L")).point(lambda p: min(255, int(p * 1.15)))
    bg.paste((20, 20, 20), (0, 0), mask)
    img = bg

    angle = rng.uniform(-1.5, 1.5)
    img = img.rotate(angle, expand=True, fillcolor=(tint, tint, tint), resample=Image.BICUBIC)

    if rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.3, 0.9)))

    if rng.random() < 0.6:
        arr = np.array(img).astype(np.int16)
        noise = np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(3, 10), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)

    return img


def make_dataset(corpus_df, text_col, out_dir, data_dir, n_per_line=2, category="synthetic_v2", seed=0):
    """Render each row n_per_line times (random font + augmentation each time). Paths in
    the returned DataFrame are relative to data_dir -- the convention the Week 4 audit
    fixed labels.csv to use consistently; keep writing rows this way so nothing regresses.
    """
    rng = random.Random(seed)
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for i, text in enumerate(corpus_df[text_col].astype(str)):
        text = text.strip()
        if not (3 <= len(text) <= 90):
            continue
        for k in range(n_per_line):
            font_name = rng.choice(list(FONTS.keys()))
            font_size = rng.randint(36, 52)
            img = augment(render_line(text, FONTS[font_name], font_size=font_size), rng)
            fname = f"{category}_{i:06d}_{k}.png"
            fpath = os.path.join(out_dir, fname)
            img.convert("RGB").save(fpath)
            rows.append({"image": os.path.relpath(fpath, data_dir), "text": text, "category": category})
    return pd.DataFrame(rows, columns=["image", "text", "category"])

In [ ]:
# Small sample of the existing headlines corpus, just to prove the renderer works
# end-to-end against real text/fonts (see the intro cell for why this stays small).
# Swap corpus_source for a clean-provenance corpus to actually scale this up.
corpus_source = pd.read_csv(os.path.join(REPO_DIR, "SI26-Week3", "corpus", "headlines.csv"), sep="\t")
sample = corpus_source[corpus_source["title"].astype(str).str.len().between(10, 60)].sample(30, random_state=0)

synthetic_v2_out = os.path.join(DATA_DIR, "raw", "synthetic_v2")
df_synthetic_v2 = make_dataset(sample, "title", synthetic_v2_out, DATA_DIR, n_per_line=3, category="synthetic_v2")
print(f"Rendered {len(df_synthetic_v2)} new synthetic images ({sample.shape[0]} lines x up to 3 variants each)")
df_synthetic_v2.head()

## Step 2: PDF Extraction (text-layer PDFs)

For PDFs you have rights to use — your own scans, open textbooks, government
publications, course materials — that have a **real text layer** (not just a scanned
image). This crops the actual rendered page to each line's position and pairs it with the
exact extracted text: real fonts, kerning, and layout, which is a different (and useful)
kind of diversity than synthetic rendering.

Uses `pypdfium2` (Apache/BSD) to render pages and `pdfplumber` (MIT) to read line
positions — both permissively licensed, per the guidance in this environment's PDF
skill, which also flags PyMuPDF/`fitz` as AGPL-3.0 (copyleft) if you're deciding between
libraries elsewhere.

**Drop your own PDFs into `DATA_DIR/user_docs/pdf/` before running the cell below.**
Pages with no text layer (scanned images) are detected and skipped, not silently
mishandled — see the OCR-assist fallback cell further down if you want to pull
unverified "silver" labels from those instead (explicitly not ground truth).

**One caveat worth knowing:** word order within a line depends on how the source PDF
encoded its glyphs, which varies by whatever tool created it. This worked correctly on
every PDF tested while building this, but spot-check a handful of outputs against the
source document before trusting a new PDF source at scale.

In [ ]:
import pdfplumber
import pypdfium2 as pdfium


def extract_pdf_lines(pdf_path, out_dir, data_dir, category="pdf_real", render_scale=3.0, min_chars=3):
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    scanned_pages = []
    base = os.path.splitext(os.path.basename(pdf_path))[0]

    pdf_render = pdfium.PdfDocument(pdf_path)
    with pdfplumber.open(pdf_path) as pdf_text:
        for page_idx, page_text in enumerate(pdf_text.pages):
            text_lines = page_text.extract_text_lines(strip=True, return_chars=False)
            if not text_lines:
                scanned_pages.append(page_idx)
                continue

            page_img = pdf_render[page_idx].render(scale=render_scale).to_pil()
            for li, line in enumerate(text_lines):
                text = line["text"].strip()
                if len(text) < min_chars:
                    continue
                pad = 4
                box = (
                    max(0, line["x0"] * render_scale - pad),
                    max(0, line["top"] * render_scale - pad),
                    min(page_img.width, line["x1"] * render_scale + pad),
                    min(page_img.height, line["bottom"] * render_scale + pad),
                )
                if box[2] - box[0] < 5 or box[3] - box[1] < 5:
                    continue
                crop = page_img.crop(box).convert("RGB")
                fname = f"{category}_{base}_{page_idx:03d}_{li:03d}.png"
                fpath = os.path.join(out_dir, fname)
                crop.save(fpath)
                rows.append({"image": os.path.relpath(fpath, data_dir), "text": text, "category": category})

    if scanned_pages:
        print(f"  {pdf_path}: {len(scanned_pages)} page(s) had no text layer (scanned?) -- "
              f"skipped: {scanned_pages}. See the OCR-assist fallback cell for that case.")
    return pd.DataFrame(rows, columns=["image", "text", "category"])


def extract_pdf_scanned_silver(pdf_path, lang="urd", render_scale=3.0):
    """OCR-assisted 'silver' labels for scanned pages, via Tesseract's Urdu model
    (`apt-get install tesseract-ocr-urd` if not already present). NOT ground truth --
    bootstrapping labels from another OCR system's output is circular. Rows come back
    tagged category='silver_needs_review' so they're easy to route to human review or
    filter out entirely before ever training on them.
    """
    import pytesseract
    pdf_render = pdfium.PdfDocument(pdf_path)
    rows = []
    for page_idx, page in enumerate(pdf_render):
        img = page.render(scale=render_scale).to_pil()
        text = pytesseract.image_to_string(img, lang=lang).strip()
        if text:
            rows.append({"page": page_idx, "text": text, "category": "silver_needs_review"})
    return pd.DataFrame(rows)

In [ ]:
import glob

pdf_input_dir = os.path.join(DATA_DIR, "user_docs", "pdf")
os.makedirs(pdf_input_dir, exist_ok=True)
pdf_files = glob.glob(os.path.join(pdf_input_dir, "*.pdf"))
print(f"Found {len(pdf_files)} PDF(s) in {pdf_input_dir}")

pdf_out = os.path.join(DATA_DIR, "raw", "pdf_v2")
df_pdf_list = [extract_pdf_lines(p, pdf_out, DATA_DIR) for p in pdf_files]
df_pdf = pd.concat(df_pdf_list, ignore_index=True) if df_pdf_list else pd.DataFrame(columns=["image", "text", "category"])
print(f"Extracted {len(df_pdf)} line images from {len(pdf_files)} PDF(s)")
df_pdf.head()

## Step 3: DOCX Extraction

Pulls paragraph text out of your own `.docx` files. DOCX is a flow document with no fixed
page layout, so there's no bounding box to crop an image from — the extracted text gets
fed through Step 1's `render_line()`/`augment()` instead, same as any other text corpus.

**Drop your own `.docx` files into `DATA_DIR/user_docs/docx/` before running.**

In [ ]:
from docx import Document


def chunk_paragraph(text, max_chars=90):
    """Split a long paragraph into line-length pieces on word boundaries."""
    words = text.split()
    chunks, cur = [], ""
    for w in words:
        if cur and len(cur) + len(w) + 1 > max_chars:
            chunks.append(cur.strip())
            cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur:
        chunks.append(cur.strip())
    return chunks


def extract_docx_lines(docx_path, max_chars=90, min_chars=3):
    doc = Document(docx_path)
    lines_out = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            lines_out.extend(chunk_paragraph(text, max_chars=max_chars))
    return [l for l in lines_out if len(l) >= min_chars]

In [ ]:
docx_input_dir = os.path.join(DATA_DIR, "user_docs", "docx")
os.makedirs(docx_input_dir, exist_ok=True)
docx_files = glob.glob(os.path.join(docx_input_dir, "*.docx"))
print(f"Found {len(docx_files)} DOCX file(s) in {docx_input_dir}")

all_docx_lines = []
for p in docx_files:
    all_docx_lines.extend(extract_docx_lines(p))
print(f"Extracted {len(all_docx_lines)} line-length chunks")

docx_out = os.path.join(DATA_DIR, "raw", "docx_v2")
if all_docx_lines:
    df_docx_corpus = pd.DataFrame({"text": all_docx_lines})
    df_docx = make_dataset(df_docx_corpus, "text", docx_out, DATA_DIR, n_per_line=1, category="docx_v2")
else:
    df_docx = pd.DataFrame(columns=["image", "text", "category"])
print(f"Rendered {len(df_docx)} images from DOCX text")
df_docx.head()

## Step 4: Merge Into `labels.csv`

Backs up the current `labels.csv` first, then appends whatever new rows exist from Steps
1–3. Safe to re-run — rerunning Steps 1–3 with the same inputs regenerates the same
filenames, and this step only adds rows for images that exist and aren't already listed.

In [ ]:
import shutil, datetime

backup_path = LABELS_PATH + f".backup-{datetime.datetime.now():%Y%m%d-%H%M%S}"
shutil.copy(LABELS_PATH, backup_path)
print("Backed up existing labels.csv to:", backup_path)

existing = pd.read_csv(LABELS_PATH)
new_rows = pd.concat([df_synthetic_v2, df_pdf, df_docx], ignore_index=True)
new_rows = new_rows[~new_rows["image"].isin(existing["image"])]  # skip anything already listed

combined = pd.concat([existing, new_rows], ignore_index=True)
combined.to_csv(LABELS_PATH, index=False)

print(f"labels.csv: {len(existing)} existing rows + {len(new_rows)} new rows = {len(combined)} total")
print(combined["category"].value_counts(dropna=False))

## Summary & Next Steps

- Re-run (or re-open) the Week 4 audited notebook — it reads this same `labels.csv`, so
  the new rows are picked up automatically. Remember its checkpoint-resume guard checks
  `dataset_size`/`train_size` against what's saved; a bigger dataset means it'll correctly
  detect the mismatch and retrain rather than resume from a stale checkpoint.
- To scale Step 1 well beyond the small test sample used above, swap in a clean-provenance
  text corpus rather than leaning further on `headlines.csv`:
  - **Wikipedia Urdu** (CC BY-SA) for general-purpose sentences.
  - **UPTI / UPTI 2.0** — 10K–120K synthetic printed Nastaliq lines, built for exactly
    this purpose (Naz et al. 2016).
  - **UTRSet-Real / UTRSet-Synth** (already partly in this project) + **UrduDoc** for
    full-page layouts (Rahman et al. 2023).
  - **MMU-OCR-21** — 602K printed images across Naskh/Nastaleeq/Tehreer fonts.
  - **FIPU-OCR-CHAR** — CC BY 4.0, on Mendeley Data, character-level.
- If you do get explicit permission from a publisher (BBC Urdu, Jang, Akhbar-e-Jahan,
  Rekhta, or anyone else) to use their content for training data, that changes the
  analysis — happy to help build a pipeline against a specific licensed export or API at
  that point.